# EXP-2026-001 / Q4-O — leakage-free morphology baseline + current-beat raw-CNN residual

| | |
|---|---|
| Spec | `experiments/specs/EXP-2026-001-q4o-leakage-free-residual-cnn.md` |
| Module | `mit-bih/q4o_leakage_free_residual.py` (all evaluation / fold / OOF / statistics logic) |
| Tests | `mit-bih/test_q4o_leakage_free_residual.py` |
| Data | `/content/drive/MyDrive/mitbih/svdb_data5.npz` — the exact file Q4-N read |
| Runtime | **GPU** (Runtime -> Change runtime type -> GPU) |

This notebook is a **wrapper only**: Drive mount, config, run, visualise, save. It
deliberately contains no evaluation logic, no fold construction, no OOF stacking, and
no statistics — all of that lives in the module so it can be unit-tested.

## Why this run exists

Q4-N built the residual CNN's offset with a function that wrote **both** the train and
the test positions of one shared array, once per fold, in sequence:

```python
sc[tr] = lr.decision_function((X[tr] - mu) / sd)   # in-sample; the next fold overwrites it
sc[te] = lr.decision_function((X[te] - mu) / sd)
```

After the last of five folds, roughly **80%** of that array holds in-sample
predictions. So `cpu_comb = 0.8445`, `boost_fix = 0.8631`, and `boost_rank = 0.8492`
are **not** baselines and **not** improvements. They are carried here only as
contaminated reference values for the Arm E diagnostic.

## The five arms

| Arm | Name | Input | Offset |
|---|---|---|---|
| A | `morph_baseline` | frozen Q4-N morphology, 17 columns | — |
| B | `raw_current_cnn` | current beat, 2 leads, nothing else | — |
| **C** | `morph_plus_raw_residual` | current beat, 2 leads | cross-fitted morph logit |
| D | `shuffled_waveform_control` | current beat, **permuted within record** | cross-fitted morph logit |
| E | `corrected_q4n_diagnostic` | Q4-N 3-beat + 2 RR channels | cross-fitted `comb` logit |

**Primary**: `C − A`. **Key negative control**: `C − D`. Arm E is diagnostic only and
must not be read as a result or a new baseline.

## Pre-registered gates (all six required for PASS)

1. `mean(C − A) >= +0.015`
2. paired record-bootstrap 95% CI lower bound `> 0`
3. `mean(C − D) > 0` and its CI lower bound `> 0`
4. at least 4 of 5 seeds positive
5. lower-tail p10 of C not worse than A by more than `0.01`
6. every leakage / reproducibility assertion passes

**NO-GO does not mean "try a Transformer."** It means keep the morphology baseline and
go back to failure-record and lower-tail analysis. PASS does not mean Transformer
either — it means port the same minimal residual structure to MIT-BIH DS1→DS2.

## 1. Mount Drive and pull the repository code

`REPO_BRANCH` must be the branch carrying this experiment. The module is imported from
the checkout, never pasted into a cell — pasted code cannot be tested.

In [ ]:
import os, sys, subprocess, time

REPO_URL    = "https://github.com/ehdbddl06001-ui/my-github-test.git"
REPO_BRANCH = "claude/exp-2026-001-q4o-leakage-free-residual"
REPO_DIR    = "/content/my-github-test"

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as exc:
    print("not Colab:", exc)
    DRIVE_ROOT = os.environ.get("MEDKOS_DRIVE_ROOT", "/content")

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR],
                   check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard",
                    f"origin/{REPO_BRANCH}"], check=True)

sys.path.insert(0, os.path.join(REPO_DIR, "mit-bih"))
import q4o_leakage_free_residual as Q

print("drive root :", DRIVE_ROOT)
print("repo commit:", Q.git_commit_sha(REPO_DIR))
print("packages   :", Q.package_versions())
print("gpu        :", Q.gpu_info())

## 2. Run the tests first

If any test fails, stop. A failing leakage assertion invalidates the run before it
starts — that is criterion 6.

In [ ]:
rc = subprocess.run(
    [sys.executable, os.path.join(REPO_DIR, "mit-bih",
                                  "test_q4o_leakage_free_residual.py")],
    capture_output=True, text=True)
print(rc.stdout[-4000:])
if rc.returncode != 0:
    print(rc.stderr[-3000:])
    raise SystemExit("tests failed — do not run the experiment")
print("tests passed")

## 3. Config

The data path is **fixed to `svdb_data5.npz`** — the exact file Q4-N read. Do not point
this at `svdb_data.npz`; it is a different, older file without `y3`/`sym`, and the
loader will refuse it. If the file is not where this cell expects it, stop and report
a blocker rather than substituting another file.

In [ ]:
DATA_PATH  = os.path.join(DRIVE_ROOT, "mitbih", "svdb_data5.npz")
PROJECT    = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
RUNS_DIR   = os.path.join(PROJECT, "runs")
REGISTRY   = os.path.join(PROJECT, "registry.jsonl")

TIMESTAMP  = time.strftime("%Y%m%dT%H%M", time.gmtime())
OUT_DIR    = os.path.join(RUNS_DIR, Q.run_dir_name(TIMESTAMP))

SEEDS      = list(Q.TRAIN_SEEDS)   # five training seeds, pre-registered
EPOCHS     = Q.DL_EPOCH
BATCH      = Q.DL_BATCH
N_BOOT     = Q.NB_BOOT
PORT_CHECK = True                  # re-score Arm A under Q4-N's LORO (fidelity check)

assert os.path.exists(DATA_PATH), (
    f"{DATA_PATH} not found. Do NOT substitute svdb_data.npz — stop and report a "
    f"blocker.")

print("data     :", DATA_PATH)
print("out dir  :", OUT_DIR)
print("seeds    :", SEEDS)
print("k-sweep  :", Q.K_SWEEP, " operating points:", Q.K_OP)
print("gates    : gain >=", Q.GATE_MIN_GAIN, "| seeds >=", Q.GATE_MIN_SEED_AGREE,
      "| lower-tail drop <=", Q.GATE_LOWER_TAIL_MAX_DROP)

## 4. Load the cohort and record its provenance

Everything the manifest needs — absolute path, SHA256, shapes, dtypes, class and
record counts — is captured here, before any modelling.

In [ ]:
cohort, provenance = Q.load_cohort(DATA_PATH)

print("file      :", provenance["file_name"])
print("sha256    :", provenance["sha256"])
print("beats     :", provenance["n_sample_labelled"], "of",
      provenance["n_sample_total"])
print("shape     :", provenance["arrays"]["beat"]["shape"],
      provenance["arrays"]["beat"]["dtype"])
print("classes   :", provenance["class_counts"])
print("records   :", provenance["n_record"], "(record == patient:",
      provenance["record_equals_patient"], ")")

rec_ok = Q.scorable_records(cohort)
burden = Q.record_burden(cohort, rec_ok)
fold_map = Q.make_fold_map(rec_ok, burden)
Q.assert_fold_map_partition(fold_map, rec_ok, Q.N_OUTER_FOLDS)
print("scorable  :", len(rec_ok), "records (MIN_S =", Q.MIN_S, ", MIN_N =", Q.MIN_N, ")")
for f in range(Q.N_OUTER_FOLDS):
    rs = sorted(r for r in rec_ok if fold_map[r] == f)
    print(f"  fold {f}: {len(rs):2d} records  {rs}")

## 5. Run

Every arm, every seed, every fold. The runner raises on any leakage violation rather
than reporting a number, so reaching the end is itself part of criterion 6.

Expect roughly 40–80 minutes on a Colab GPU for five seeds × four neural arms ×
five folds.

In [ ]:
log = Q.RunLog()
result = Q.run_experiment(
    cohort, provenance, OUT_DIR,
    seeds=SEEDS, epochs=EPOCHS, batch=BATCH, n_boot=N_BOOT,
    port_check=PORT_CHECK, smoke=False, log=log)
print("\nverdict:", result["gates"]["verdict"])

## 6. Read the result

Read the primary comparison and the negative control together. A `C − A` gain that is
not also a `C − D` gain is the waveform-shuffle control telling you the effect is not
coming from beat-level waveform information.

In [ ]:
import json

print("=" * 78)
print("arms — seed-averaged record-level k-sweep achievement mean")
print("=" * 78)
for arm, m in result["arms"].items():
    s = m["seed_averaged_ksw"]
    print(f"  {arm:<28} {s['mean']:.4f}  p10 {s['p10']:.4f}  "
          f"worst {s['worst']:.4f} (rec {s['worst_record']})  "
          f"seed sd {m['ksw_seed_std']:.4f}")

print("\n" + "=" * 78)
print("contrasts — paired on the same record and the same seed")
print("=" * 78)
for name, c in result["contrasts"].items():
    b, h = c["record_bootstrap"], c["hierarchical_bootstrap"]
    print(f"  {name:<22} {b['mean']:+.4f} [{b['ci_low']:+.4f}, {b['ci_high']:+.4f}] "
          f"record boot | [{h['ci_low']:+.4f}, {h['ci_high']:+.4f}] hierarchical | "
          f"seeds + {c['positive_seed_count']}/{len(SEEDS)}")

print("\n" + "=" * 78)
print("pre-registered gates")
print("=" * 78)
for k, v in result["gates"]["checks"].items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")
print(f"\n  VERDICT: {result['gates']['verdict']}")
print(f"  NEXT   : {result['gates']['next_step']}")

### Arm E — the Q4-N diagnostic

Compare Arm E against `comb_baseline_diagnostic` (the clean analogue of Q4-N's
`cpu_comb`), **not** against Arm A: Arm E's offset is the 28-column `comb` set while
Arm A's is the 17-column `morph` set, so `E − A` would mix the feature change with
the residual.

Q4-N's numbers also came from a different split, so do not subtract them from these.
Read the direction, not the difference.

In [ ]:
e = result["arm_E_diagnostic"]
print("Arm E (corrected, cross-fitted offset) :", f"{e['arm_E_seed_averaged_ksw']:.4f}")
print("comb logistic baseline (clean cpu_comb):",
      f"{e['comb_baseline_seed_averaged_ksw']:.4f}")
r_ = e["residual_effect_isolated"]
print("isolated residual effect               :",
      f"{r_['mean']:+.4f} [{r_['ci_low']:+.4f}, {r_['ci_high']:+.4f}]")
print("\nQ4-N CONTAMINATED reference values (NOT baselines):")
for k, v in e["q4n_contaminated_reference"].items():
    print(f"  {k:<12} {v}")
print("\n" + e["interpretation"])
print("\n" + e["protocol_note"])

### Porting-fidelity check

Re-scores Arm A's feature matrix under Q4-N's original leave-one-record-out protocol
and compares against Q4-N's reported `morph` k-sweep of `0.8361`. This checks the
**feature port**, not the hypothesis. A large delta means the ported morphology code
diverged from Q4-N and the run should not be interpreted until that is resolved.

In [ ]:
man = json.load(open(os.path.join(OUT_DIR, "manifest.json")))
print(json.dumps(man["morph_port_check"], indent=2))

## 7. Figures

The module writes `figures/arms_ksweep.png` and `figures/contrasts.png` into the run
bundle. Anything drawn below is presentation only — no number is recomputed here.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

for name in ("arms_ksweep.png", "contrasts.png"):
    p = os.path.join(OUT_DIR, "figures", name)
    if os.path.exists(p):
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.imshow(mpimg.imread(p))
        ax.axis("off")
        ax.set_title(name, fontsize=9)
        plt.show()

In [ ]:
# Per-record C - A, so the lower tail is visible rather than summarised away.
import numpy as np

pred = np.load(os.path.join(OUT_DIR, "predictions.npz"))
la = pred[f"logit_{Q.ARM_A}"][0]      # Arm A is deterministic across seeds
lc = pred[f"logit_{Q.ARM_C}"][0]      # seed 0

ksw_a = Q.per_record_metrics(la, cohort, rec_ok)["ksw"]
ksw_c = Q.per_record_metrics(lc, cohort, rec_ok)["ksw"]
recs = sorted(ksw_a)
d = np.array([ksw_c[r] - ksw_a[r] for r in recs])
order = np.argsort(d)

fig, ax = plt.subplots(figsize=(10, 3.4))
ax.bar(np.arange(len(recs)), d[order],
       color=["tab:red" if v < 0 else "tab:blue" for v in d[order]])
ax.axhline(0, color="k", lw=0.8)
ax.axhline(Q.GATE_MIN_GAIN, color="tab:green", ls="--", lw=0.8, label="gate +0.015")
ax.set_xticks(np.arange(len(recs)))
ax.set_xticklabels([recs[i] for i in order], rotation=90, fontsize=6)
ax.set_xlabel("record")
ax.set_ylabel("C - A (k-sweep, seed 0)")
ax.legend(fontsize=7)
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print(f"records where C beats A : {int((d > 0).sum())} / {len(d)}")
print(f"worst record            : {recs[int(np.argmin(d))]}  ({d.min():+.4f})")

## 8. Registry

`registry.jsonl` gets **one line per measured run**, written from the result that was
actually produced above. Never write a value here by hand.

In [ ]:
record = {
    "run_id": Q.run_dir_name(TIMESTAMP),
    "experiment_id": Q.EXPERIMENT_ID,
    "arm_id": Q.ARM_ID,
    "primary_metric": result["primary_metric"],
    "primary_value": result["contrasts"]["C_minus_A"]["record_bootstrap"]["mean"],
    "primary_ci": [result["contrasts"]["C_minus_A"]["record_bootstrap"]["ci_low"],
                   result["contrasts"]["C_minus_A"]["record_bootstrap"]["ci_high"]],
    "negative_control": result["contrasts"]["C_minus_D"]["record_bootstrap"]["mean"],
    "verdict": result["gates"]["verdict"],
    "conclusion": (f"C-A {result['contrasts']['C_minus_A']['record_bootstrap']['mean']:+.4f}, "
                   f"C-D {result['contrasts']['C_minus_D']['record_bootstrap']['mean']:+.4f}, "
                   f"{result['gates']['verdict']}"),
    "run_folder": OUT_DIR,
    "data_sha256": provenance["sha256"],
    "git_commit": Q.git_commit_sha(REPO_DIR),
}
Q.append_registry(REGISTRY, record)
print(json.dumps(record, indent=2))

## 9. Bring the run back into GitHub

1. Save this executed notebook to `notebooks/quest47_q4o_leakage_free_residual_cnn.ipynb`
   and commit it — an unexecuted notebook is not evidence.
2. Ingest the measured result:

```bash
python pipelines/ingest_run.py \
    --results <run_dir>/result.json \
    --notebook notebooks/quest47_q4o_leakage_free_residual_cnn.ipynb
```

3. Register the run bundle in `research/ASSETS.md` (path, not a move).
4. Open the review PR with the exact commands and any deviations.

Do **not** update `research/PROJECT_STATE.md` with a new baseline until the design
owner has read the executed notebook and the measured result. Until then this
experiment has no outcome.

In [ ]:
print("run bundle:", OUT_DIR)
for root, dirs, files in os.walk(OUT_DIR):
    for f_ in sorted(files):
        p = os.path.join(root, f_)
        print(f"  {os.path.relpath(p, OUT_DIR):<48} {os.path.getsize(p):>10,d} bytes")
Q.verify_bundle(OUT_DIR)
print("\nbundle schema verified")